### Bootcamp: Week 2 Exercise Deep Research with clarifying questions

In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, gen_trace_id, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content

In [ ]:
load_dotenv(override=True)

MODEL = "gpt-5-nano"

In [ ]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model=MODEL,
    model_settings=ModelSettings(tool_choice="required"),
)

In [ ]:
HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model=MODEL,
    output_type=WebSearchPlan,
)

In [ ]:
CLARIFY_INSTRUCTIONS = (
    "You are a research intake assistant. Given the user's research query, "
    "ask the minimum useful set of clarifying questions. "
    "Return between 1 and 6 concise questions, as a plain list."
)

class ClarifyingQuestions(BaseModel):
    questions: list[str] = Field(
        description="Clarifying questions to improve research scope and output quality."
    )

clarification_agent = Agent(
    name="ClarificationAgent",
    instructions=CLARIFY_INSTRUCTIONS,
    model=MODEL,
    output_type=ClarifyingQuestions,
)

def build_enriched_query(query: str, questions: list[str], answers_text: str) -> str:
    answers_text = (answers_text or "").strip()
    if not questions or not answers_text:
        return query
    lines = [f"Original query: {query}", "", "Clarifying questions:"]
    for i, q in enumerate(questions, start=1):
        lines.append(f"Q{i}: {q}")
    lines += ["", "User responses to clarifying questions:", answers_text]
    return "\n".join(lines)

async def get_clarifying_questions(query: str) -> ClarifyingQuestions:
    result = await Runner.run(
        clarification_agent,
        f"User research query:\n{query}",
    )
    return result.final_output_as(ClarifyingQuestions)

In [ ]:
@function_tool
def send_email(subject: str, html_body: str) -> dict:
    """
    Send out an email with the given subject and HTML body.
    Intended for use by an agent that formats reports into professional HTML emails.
    """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("mutuma212@gmail.com")  # Verified "from" email for SendGrid
    to_email = To("mutuma212@gmail.com")       # Recipient (update to your address)
    content = Content("text/html", html_body)  # Content is explicitly HTML
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [ ]:
INSTRUCTIONS = (
    "You are an expert assistant responsible for taking a detailed report and sending it out as a professional HTML email using your provided tool. "
    "You will receive a research report as input. "
    "Your responsibilities:\n"
    "- Convert the entire report into clean, well-structured HTML, ensuring the content is visually appealing (headings, paragraphs, etc).\n"
    "- Craft a clear, concise, and appropriate subject line based on the report's content.\n"
    "- Use your send_email tool to dispatch ONE email containing the HTML-formatted report and the chosen subject.\n"
    "Consider professionalism and clarity in all formatting."
)

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model=MODEL,
)

In [ ]:
INSTRUCTIONS = (
    "You are an expert research assistant collaborating as part of a multi-agent workflow to produce high-quality written reports.\n"
    "You will receive the original research query as well as search results and supporting material gathered by automated search agents.\n"
    "First, create an outline for a comprehensive report, listing main sections and key points to ensure logical organization and complete coverage of the topic.\n"
    "Then, using all available information, write a detailed and cohesive report in markdown format. Your report should synthesize the material thoughtfully, incorporate critical analysis, and be formatted for clarity and professional presentation suitable for an expert audience.\n"
    "Make the report longform (target at least 1000 words), using headings, bullet points, and other markdown features as appropriate to structure the content."
)

class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report in markdown format")
    follow_up_questions: list[str] = Field(description="Suggested follow-up research questions")

writer_agent = Agent(
    name="Writer agent",
    instructions=INSTRUCTIONS,
    model=MODEL,
    output_type=ReportData,
)

### Showtime!

In [ ]:
class ResearchManager:

    async def run(self, query: str, questions: list[str] | None = None, answers_text: str = ""):
        """Run deep research. Clarifications are optional."""
        trace_id = gen_trace_id()
        questions = questions or []
        enriched = build_enriched_query(query, questions, answers_text)
        with trace("Research trace", trace_id=trace_id):
            print(f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}")
            yield f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}"
            yield "Starting research..."
            search_plan = await self.plan_searches(enriched)
            yield "Searches planned, starting to search..."
            search_results = await self.perform_searches(search_plan)
            yield "Searches complete, writing report..."
            report = await self.write_report(enriched, search_results)
            yield "Report written, sending email..."
            await self.send_email(report)
            yield "Email sent, research complete"
            yield report.markdown_report

    async def plan_searches(self, enriched: str) -> WebSearchPlan:
        """ Plan the searches to perform for the query """
        print("Planning searches...")
        result = await Runner.run(
            planner_agent,
            f"Query: {enriched}",
        )
        print(f"Will perform {len(result.final_output.searches)} searches")
        return result.final_output_as(WebSearchPlan)

    async def perform_searches(self, search_plan: WebSearchPlan) -> list[str]:
        """ Perform the searches to perform for the query """
        print("Searching...")
        num_completed = 0
        tasks = [asyncio.create_task(self.search(item)) for item in search_plan.searches]
        results = []
        for task in asyncio.as_completed(tasks):
            result = await task
            if result is not None:
                results.append(result)
            num_completed += 1
            print(f"Searching... {num_completed}/{len(tasks)} completed")
        print("Finished searching")
        return results

    async def search(self, item: WebSearchItem) -> str | None:
        """ Perform a search for the query """
        input = f"Search term: {item.query}\nReason for searching: {item.reason}"
        
        result = await Runner.run(
                search_agent,
                input,
            )
        return str(result.final_output)

    async def write_report(self, enriched: str, search_results: list[str]) -> ReportData:
        """ Write the report for the query """
        print("Thinking about report...")
        input = f"Original query: {enriched}\nSummarized search results: {search_results}"
        result = await Runner.run(
            writer_agent,
            input,
        )

        print("Finished writing report")
        return result.final_output_as(ReportData)
    
    async def send_email(self, report: ReportData) -> None:
        print("Writing email...")
        await Runner.run(
            email_agent,
            report.markdown_report,
        )
        print("Email sent")

In [ ]:
import gradio as gr


async def on_query_submit(query: str, state: dict | None):
    query = (query or "").strip()
    if not query:
        return "Enter a research topic first.", state

    cq = await get_clarifying_questions(query)
    qs = [q.strip() for q in cq.questions if q.strip()]

    state = {"query": query, "questions": qs}
    if not qs:
        return "_No clarifying questions generated. You can run directly._", state

    md = "\n\n".join(f"**{i}.** {q}" for i, q in enumerate(qs, 1))
    return md, state


async def on_run(query: str, state: dict | None, answers_text: str):
    query = (query or "").strip()
    if not query:
        yield "Please enter a query first."
        return

    questions = []
    if state and state.get("query") == query:
        questions = state.get("questions", [])

    # If answers are empty, pipeline proceeds with raw query.
    async for chunk in ResearchManager().run(query, questions, answers_text):
        yield chunk


with gr.Blocks(theme=gr.themes.Default(primary_hue="sky")) as ui:
    gr.Markdown("# Deep Research")

    session = gr.State(value=None)

    query_textbox = gr.Textbox(
        label="What topic would you like to research?",
        placeholder="Type query and press Enter to generate clarifying questions (optional).",
    )
    questions_md = gr.Markdown(label="Clarifying questions (optional)")

    answers_box = gr.Textbox(
        label="Your responses (optional)",
        lines=8,
        placeholder="If you want, answer the clarifying questions here. If left blank, research runs on your original query.",
    )

    run_btn = gr.Button("Run research", variant="primary")
    report = gr.Markdown(label="Report")

    query_textbox.submit(
        fn=on_query_submit,
        inputs=[query_textbox, session],
        outputs=[questions_md, session],
    )

    run_btn.click(
        fn=on_run,
        inputs=[query_textbox, session, answers_box],
        outputs=report,
    )

ui.launch(inbrowser=True)